# Customer Success & Retention Operations Guide

**Complete guide for customer onboarding, engagement, retention, and reducing churn.**

Learn how to build lasting customer relationships and grow with intent.


## Customer Success Framework

### Customer Lifecycle

```
Awareness → Acquisition → Activation → Retention → Expansion → Advocacy
   ↓           ↓            ↓           ↓           ↓          ↓
  Marketing  Signup       First Use   Engagement  Upsell    Referrals
```

### Success Metrics by Stage

```
Acquisition:
├─ Sign-up rate
├─ Cost per acquisition (CAC)
└─ Source attribution

Activation (First 7 days):
├─ % users completing onboarding
├─ % users taking first action
└─ % users reaching 'aha' moment

Retention (Month-over-month):
├─ Day-1 retention rate
├─ Day-7 retention rate
├─ Day-30 retention rate
├─ Monthly churn rate

Expansion:
├─ Upgrade rate (free → pro)
├─ Upsell rate (pro → enterprise)
├─ Feature adoption rate

Advocacy:
├─ Net Promoter Score (NPS)
├─ Referral rate
└─ Customer satisfaction (CSAT)
```

---

## Onboarding Strategy

### Onboarding Flow

```python
# ai-projects/onboarding/onboarding_service.py
from enum import Enum
from datetime import datetime, timedelta

class OnboardingStage(Enum):
    WELCOME = "welcome"
    SETUP_PROFILE = "setup_profile"
    FEATURE_TOUR = "feature_tour"
    FIRST_CHAT = "first_chat"
    SETTINGS = "settings"
    COMPLETE = "complete"

class OnboardingService:
    def __init__(self, db):
        self.db = db

    def get_onboarding_status(self, user_id: str) -> dict:
        """Get user's onboarding progress."""
        query = """
        SELECT stage, completed_at, skipped_stages
        FROM user_onboarding
        WHERE user_id = %s
        """
        result = self.db.execute(query, (user_id,))
        row = result.one_or_none()

        if not row:
            return self._initialize_onboarding(user_id)

        return {
            "current_stage": OnboardingStage(row['stage']),
            "progress": self._calculate_progress(row['skipped_stages']),
            "completed_at": row['completed_at']
        }

    def _initialize_onboarding(self, user_id: str) -> dict:
        """Initialize onboarding for new user."""
        self.db.insert("user_onboarding", {
            "user_id": user_id,
            "stage": OnboardingStage.WELCOME.value,
            "created_at": datetime.now()
        })

        # Send welcome email
        self._send_email(user_id, "welcome")

        return {
            "current_stage": OnboardingStage.WELCOME,
            "progress": 0,
            "completed_at": None
        }

    def advance_stage(self, user_id: str):
        """Move to next onboarding stage."""
        current = self.get_onboarding_status(user_id)
        stages = list(OnboardingStage)
        current_idx = stages.index(current['current_stage'])

        if current_idx < len(stages) - 1:
            next_stage = stages[current_idx + 1]
            self.db.update(
                "user_onboarding",
                {"stage": next_stage.value},
                {"user_id": user_id}
            )

            # Send stage-specific email
            self._send_stage_email(user_id, next_stage)

            return next_stage
        else:
            # Onboarding complete
            self.db.update(
                "user_onboarding",
                {"completed_at": datetime.now()},
                {"user_id": user_id}
            )
            self._send_email(user_id, "onboarding_complete")

    def skip_stage(self, user_id: str):
        """Skip current stage."""
        self.db.update(
            "user_onboarding",
            {"skipped_stages": [OnboardingStage.current_stage.value]},
            {"user_id": user_id}
        )
        self.advance_stage(user_id)
```

### Onboarding Emails

```yaml
# config/onboarding_emails.yaml
onboarding_sequence:
    welcome:
        delay: 0m
        subject: "Welcome to Aria! 🚀"
        template: welcome.html
        cta: "Start Tour"

    feature_intro:
        delay: 1h
        subject: "Here's what you can do with Aria"
        template: feature_intro.html
        content:
            - Natural language commands
            - Multiple AI models
            - Semantic memory
            - Custom instructions

    first_use_reminder:
        delay: 1d
        subject: "Let's create your first message 💬"
        template: first_use_reminder.html
        content:
            - Example prompts
            - Tips for best results
            - Common use cases

    tier_intro:
        delay: 3d
        subject: "Upgrade to Pro for unlimited access"
        template: tier_intro.html
        show_if: tier == "free"
        content:
            - Feature comparison
            - Special offer (20% off first month)

    week_1_digest:
        delay: 7d
        subject: "Your Aria week recap 📊"
        template: week_digest.html
        content:
            - Messages sent
            - Favorite features
            - Usage tips
```

---

## Engagement & Retention

### Engagement Scoring

```python
# shared/engagement_score.py
from datetime import datetime, timedelta

class EngagementScore:
    """Calculate user engagement score (0-100)."""

    def __init__(self, db, user_id: str):
        self.db = db
        self.user_id = user_id

    def calculate(self) -> float:
        """Calculate engagement score."""
        # Last 30 days activity
        last_30d = datetime.now() - timedelta(days=30)

        metrics = {
            "messages_sent": self._count_messages(last_30d),
            "days_active": self._count_active_days(last_30d),
            "features_used": self._count_features_used(last_30d),
            "avg_daily_session": self._avg_session_length(last_30d),
            "model_variety": self._model_diversity(last_30d)
        }

        # Calculate weighted score
        score = (
            metrics["messages_sent"] * 0.3 +
            metrics["days_active"] * 0.25 +
            metrics["features_used"] * 0.2 +
            metrics["avg_daily_session"] * 0.15 +
            metrics["model_variety"] * 0.1
        )

        # Normalize to 0-100
        return min(100, max(0, score))

    def _count_messages(self, since: datetime) -> float:
        """Normalize message count."""
        query = "SELECT COUNT(*) FROM messages WHERE user_id = %s AND created_at > %s"
        count = self.db.execute(query, (self.user_id, since)).scalar()
        # 100 messages = max score
        return min(100, (count / 100) * 100)

    def _count_active_days(self, since: datetime) -> float:
        """Days with at least one action."""
        query = """
        SELECT COUNT(DISTINCT DATE(created_at))
        FROM messages
        WHERE user_id = %s AND created_at > %s
        """
        days = self.db.execute(query, (self.user_id, since)).scalar()
        # 30 days = max score
        return min(100, (days / 30) * 100)

    def _count_features_used(self, since: datetime) -> float:
        """Different features used."""
        query = """
        SELECT COUNT(DISTINCT feature_name)
        FROM usage_events
        WHERE user_id = %s AND created_at > %s
        """
        features = self.db.execute(query, (self.user_id, since)).scalar()
        # 10+ features = max score
        return min(100, (features / 10) * 100)

def segment_users_by_engagement() -> dict:
    """Segment users for targeted retention."""
    query = """
    SELECT
        user_id,
        CASE
            WHEN engagement_score >= 75 THEN 'highly_engaged'
            WHEN engagement_score >= 50 THEN 'engaged'
            WHEN engagement_score >= 25 THEN 'at_risk'
            ELSE 'inactive'
        END as segment,
        COUNT(*) as user_count
    FROM user_engagement_scores
    WHERE calculated_at > NOW() - INTERVAL '7 days'
    GROUP BY segment
    """
    return db.execute(query).fetchall()
```

### At-Risk User Detection

```python
# scripts/detect_at_risk_users.py
from datetime import datetime, timedelta

class AtRiskDetector:
    """Identify users likely to churn."""

    CHURN_SIGNALS = {
        "no_activity_7d": -20,
        "no_activity_14d": -30,
        "decreased_activity": -15,
        "error_rate_high": -10,
        "support_tickets": -5,
        "plan_downgrade": -25
    }

    def __init__(self, db):
        self.db = db

    def detect_at_risk(self, threshold: int = -50) -> list:
        """Find users with churn signals."""
        users = self.db.execute("SELECT user_id FROM users WHERE status = 'active'").fetchall()
        at_risk = []

        for user_id in users:
            risk_score = self._calculate_risk_score(user_id)

            if risk_score < threshold:
                at_risk.append({
                    "user_id": user_id,
                    "risk_score": risk_score,
                    "signals": self._get_signals(user_id),
                    "recommended_action": self._recommend_action(risk_score)
                })

        return sorted(at_risk, key=lambda x: x['risk_score'])

    def _calculate_risk_score(self, user_id: str) -> int:
        """Calculate risk score based on signals."""
        score = 0

        # Check last activity
        query = "SELECT MAX(created_at) FROM messages WHERE user_id = %s"
        last_activity = self.db.execute(query, (user_id,)).scalar()

        if not last_activity:
            score += self.CHURN_SIGNALS["no_activity_14d"]
        elif (datetime.now() - last_activity).days > 14:
            score += self.CHURN_SIGNALS["no_activity_14d"]
        elif (datetime.now() - last_activity).days > 7:
            score += self.CHURN_SIGNALS["no_activity_7d"]

        # Check for errors
        query = """
        SELECT COUNT(*) FROM errors
        WHERE user_id = %s AND created_at > NOW() - INTERVAL '7 days'
        """
        errors = self.db.execute(query, (user_id,)).scalar()
        if errors > 5:
            score += self.CHURN_SIGNALS["error_rate_high"]

        return score

    def _recommend_action(self, risk_score: int) -> str:
        """Recommend retention action."""
        if risk_score < -75:
            return "Personal outreach + offer discount"
        elif risk_score < -50:
            return "Re-engagement email + feature tips"
        else:
            return "Monitor weekly + send tips"
```

### Retention Campaigns

```yaml
# config/retention_campaigns.yaml
campaigns:
    win_back_inactive:
        trigger: no_activity_30d
        delay: 1d
        emails:
            - subject: "We miss you! 👋"
              template: miss_you.html
              content:
                  - What's new in Aria
                  - Special comeback offer
                  - Personal support offer
            - subject: "Last chance: 50% off Pro tier"
              delay: 3d
              template: special_offer.html
        success_metric: reactivation_rate

    at_risk_intervention:
        trigger: engagement_score < 30
        delay: 0
        emails:
            - subject: "How can we help you?"
              template: help_offer.html
              action: schedule_call
            - subject: "Tips to get more value from Aria"
              delay: 2d
              template: tips.html
        success_metric: engagement_improvement

    upgrade_push:
        trigger: tier == "free" AND messages_sent > 50
        delay: 7d
        emails:
            - subject: "Unlock unlimited with Pro 🚀"
              template: upgrade_offer.html
              offer: "20% off first month"
        success_metric: upgrade_rate

churn_recovery:
    trigger: subscription_cancelled
    delay: 1h
    steps:
        - "Exit survey (learn why)"
        - "Discount offer (win back)"
        - "Satisfaction followup (6 months)"
```


## Customer Health Dashboards

### Key Health Metrics

```python
# scripts/customer_health_dashboard.py
import pandas as pd

class CustomerHealthDashboard:
    def __init__(self, db):
        self.db = db

    def get_health_metrics(self) -> dict:
        """Get key health metrics."""
        return {
            "retention": self._get_retention_metrics(),
            "engagement": self._get_engagement_metrics(),
            "expansion": self._get_expansion_metrics(),
            "churn_risks": self._get_churn_risks()
        }

    def _get_retention_metrics(self) -> dict:
        """Month-over-month retention."""
        query = """
        WITH cohorts AS (
            SELECT
                DATE_TRUNC('month', created_at) as cohort_month,
                user_id
            FROM users
        ),
        activity AS (
            SELECT
                DATE_TRUNC('month', created_at) as activity_month,
                user_id
            FROM messages
            GROUP BY DATE_TRUNC('month', created_at), user_id
        )
        SELECT
            c.cohort_month,
            COUNT(DISTINCT c.user_id) as cohort_size,
            COUNT(DISTINCT CASE WHEN a.activity_month = c.cohort_month THEN c.user_id END) as m0,
            COUNT(DISTINCT CASE WHEN a.activity_month = c.cohort_month + INTERVAL '1 month' THEN c.user_id END) as m1,
            COUNT(DISTINCT CASE WHEN a.activity_month = c.cohort_month + INTERVAL '2 months' THEN c.user_id END) as m2
        FROM cohorts c
        LEFT JOIN activity a ON c.user_id = a.user_id
        GROUP BY c.cohort_month
        ORDER BY c.cohort_month DESC
        """

        df = pd.read_sql(query, self.db.engine)
        return df.to_dict('records')

    def _get_engagement_metrics(self) -> dict:
        """Segment users by engagement."""
        query = """
        SELECT
            CASE
                WHEN engagement_score >= 75 THEN 'highly_engaged'
                WHEN engagement_score >= 50 THEN 'engaged'
                WHEN engagement_score >= 25 THEN 'at_risk'
                ELSE 'inactive'
            END as segment,
            COUNT(*) as user_count,
            ROUND(AVG(engagement_score)) as avg_score
        FROM user_engagement_scores
        WHERE calculated_at > NOW() - INTERVAL '7 days'
        GROUP BY segment
        ORDER BY avg_score DESC
        """

        return pd.read_sql(query, self.db.engine).to_dict('records')

    def _get_expansion_metrics(self) -> dict:
        """Upgrade and expansion rates."""
        query = """
        SELECT
            COUNT(CASE WHEN upgraded_at > NOW() - INTERVAL '30 days' THEN 1 END) as upgrades_30d,
            COUNT(CASE WHEN upgraded_at > NOW() - INTERVAL '90 days' THEN 1 END) as upgrades_90d,
            ROUND(100.0 * COUNT(CASE WHEN upgraded_at > NOW() - INTERVAL '30 days' THEN 1 END) /
                  COUNT(CASE WHEN tier = 'free' AND created_at < NOW() - INTERVAL '30 days' THEN 1 END), 2)
                  as upgrade_rate_pct
        FROM user_subscriptions
        """

        return pd.read_sql(query, self.db.engine).iloc[0].to_dict()

    def _get_churn_risks(self) -> list:
        """Top 10 at-risk users."""
        query = """
        SELECT
            user_id,
            email,
            tier,
            engagement_score,
            days_since_activity,
            created_at
        FROM users u
        LEFT JOIN user_engagement_scores e ON u.user_id = e.user_id
        WHERE u.status = 'active'
        ORDER BY engagement_score ASC
        LIMIT 10
        """

        return pd.read_sql(query, self.db.engine).to_dict('records')
```

## Best Practices

### Onboarding

- [ ] First 5 minutes: Show immediate value
- [ ] First day: Complete setup wizard
- [ ] First week: Feature deep dive
- [ ] First month: Use case mastery
- [ ] Measure: % reaching "aha moment" by day 7

### Engagement

- [ ] Weekly digest emails
- [ ] In-app tips and tutorials
- [ ] Feature announcements
- [ ] Usage milestones (100 messages, etc.)
- [ ] Personalized recommendations

### Retention

- [ ] Monitor engagement weekly
- [ ] Target interventions at-risk users
- [ ] Win-back campaigns for inactive
- [ ] Exclusive perks for long-term users
- [ ] Ask for feedback (surveys, NPS)

### Expansion

- [ ] Upgrade prompts at usage limits
- [ ] Feature comparison charts
- [ ] Limited-time offers
- [ ] Case studies from similar users
- [ ] Personal sales calls for enterprise
